# Aura AI — Fine-Tune Existing Qwen3 0.6B with Korean SFT

This run fine-tunes the existing pretrained Qwen3-0.6B model with a QLoRA adapter. It does not initialize a new language model. Training loss is applied only to assistant responses; user and system prompt tokens are masked.


In [ ]:
import os, json, torch, shutil
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen3-0.6B"
INPUT_ROOT = Path("/kaggle/input")
train_candidates = list(INPUT_ROOT.rglob("train.jsonl"))
data_candidates = [p.parent for p in train_candidates if (p.parent / "valid.jsonl").is_file()]
if not data_candidates:
    raise RuntimeError("Existing Aura Korean SFT dataset is not mounted. Inputs: " + repr([p.name for p in INPUT_ROOT.iterdir()]))
DATA_ROOT = data_candidates[0]
OUTPUT = Path("/kaggle/working/aura-qwen3-0.6b-korean-qlora")
MAX_LENGTH = 768
SEED = 20261125

assert torch.cuda.is_available(), "Enable the Kaggle NVIDIA GPU accelerator."
print({"base_model": BASE_MODEL, "gpu": torch.cuda.get_device_name(0), "vram_gib": round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2), "dataset": str(DATA_ROOT)}, flush=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
raw = load_dataset("json", data_files={
    "train": str(DATA_ROOT / "train.jsonl"),
    "validation": str(DATA_ROOT / "valid.jsonl"),
})
print("DATASET_ROWS", {k: len(v) for k, v in raw.items()}, flush=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def normalize_token_ids(value):
    # Recent Transformers may return BatchEncoding instead of a plain token list.
    if isinstance(value, dict) or hasattr(value, "data"):
        try:
            value = value["input_ids"]
        except Exception:
            pass
    if hasattr(value, "tolist"):
        value = value.tolist()
    if isinstance(value, tuple):
        value = list(value)
    if isinstance(value, list) and value and isinstance(value[0], list):
        value = value[0]
    return list(value)

def encode_row(row):
    messages = [
        {"role": m.get("role"), "content": m.get("content", "")}
        for m in row.get("messages", [])
        if m.get("role") in {"system", "user", "assistant"} and isinstance(m.get("content"), str)
    ]
    if not messages or not any(m["role"] == "assistant" and m["content"].strip() for m in messages):
        return {"input_ids": [], "attention_mask": [], "labels": []}

    full_ids = normalize_token_ids(tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False, enable_thinking=False
    ))
    labels = [-100] * len(full_ids)

    # Supervise assistant turns only; system/user prompt tokens are masked.
    for index, message in enumerate(messages):
        if message["role"] != "assistant" or not message["content"].strip():
            continue
        prefix_ids = normalize_token_ids(tokenizer.apply_chat_template(
            messages[:index], tokenize=True, add_generation_prompt=True, enable_thinking=False
        ))
        completed_ids = normalize_token_ids(tokenizer.apply_chat_template(
            messages[:index + 1], tokenize=True, add_generation_prompt=False, enable_thinking=False
        ))
        start = len(prefix_ids)
        if completed_ids[:start] != prefix_ids:
            common = 0
            limit = min(len(prefix_ids), len(completed_ids))
            while common < limit and prefix_ids[common] == completed_ids[common]:
                common += 1
            start = common
        end = min(len(completed_ids), len(full_ids))
        for position in range(start, end):
            labels[position] = full_ids[position]

    full_ids = full_ids[-MAX_LENGTH:]
    labels = labels[-MAX_LENGTH:]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

tokenized = raw.map(
    encode_row,
    remove_columns=raw["train"].column_names,
    desc="Tokenizing existing Aura conversations with assistant-only loss",
)
tokenized = tokenized.filter(
    lambda row: len(row["input_ids"]) >= 8 and any(label != -100 for label in row["labels"])
)
print("TOKENIZED_ROWS", {k: len(v) for k, v in tokenized.items()}, flush=True)

class PadCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id
    def __call__(self, features):
        width = max(len(item["input_ids"]) for item in features)
        ids = torch.full((len(features), width), self.pad_id, dtype=torch.long)
        mask = torch.zeros((len(features), width), dtype=torch.long)
        labels = torch.full((len(features), width), -100, dtype=torch.long)
        for i, item in enumerate(features):
            n = len(item["input_ids"])
            ids[i, :n] = torch.tensor(item["input_ids"], dtype=torch.long)
            mask[i, :n] = 1
            labels[i, :n] = torch.tensor(item["labels"], dtype=torch.long)
        return {"input_ids": ids, "attention_mask": mask, "labels": labels}

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    torch_dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=str(OUTPUT),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1.5e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    eval_steps=300,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=2,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    group_by_length=True,
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=PadCollator(tokenizer.pad_token_id),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)
result = trainer.train()
metrics = trainer.evaluate()
OUTPUT.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(OUTPUT, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT)
trainer.save_state()
metrics_payload = {
    "base_model": BASE_MODEL,
    "train_rows": len(tokenized["train"]),
    "validation_rows": len(tokenized["validation"]),
    "train_metrics": result.metrics,
    "evaluation_metrics": metrics,
    "epochs": 2,
    "max_length": MAX_LENGTH,
    "method": "QLoRA NF4, rank 16, assistant-only loss",
}
Path("/kaggle/working/aura-qwen3-training-metrics.json").write_text(
    json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("AURA_QWEN3_QLORA_COMPLETE", json.dumps(metrics), flush=True)

trainer.model.config.use_cache = True
prompt = [
    {"role": "system", "content": "너는 Aura AI다. 정확하고 자연스러운 한국어로 답한다."},
    {"role": "user", "content": "너를 소개하고, 파인튜닝에 사용한 방식을 간단히 설명해 줘."},
]
inputs = tokenizer.apply_chat_template(
    prompt, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt"
).to("cuda")
with torch.inference_mode():
    generated = trainer.model.generate(
        inputs, max_new_tokens=120, do_sample=False,
        pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
    )
print("SMOKE_TEST", tokenizer.decode(generated[0][inputs.shape[-1]:], skip_special_tokens=True), flush=True)

archive = shutil.make_archive("/kaggle/working/aura-qwen3-0.6b-korean-qlora", "zip", OUTPUT)
print("ADAPTER_ARCHIVE", archive, Path(archive).stat().st_size, flush=True)
